# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MissNaliaka/SEO-Content-Opportunity-Scoring/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*


Finding 1 - "What Predicts Health?" (Random Forest feature importance, p.27)
The paper defines Health Score as (Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts).
The Random Forest's top predictors of that same score are Position (43%), Impressions (32%), Scroll Depth (15%), and CTR (8%) - the exact four components the score is built from. Does the validation design carry the claim? No — the paper says the model is "holdout-tested," but a holdout split doesn't rescue this: if the label is a weighted sum of the features, a model will score well on any holdout by re-deriving the same formula, not by learning anything new. High out-of-sample accuracy here proves the arithmetic is consistent, not that these are real drivers of quality. The paper does flag this itself ("importance is descriptive rather than causal") — a good instinct — but a reader skimming the bar chart could still walk away thinking "improve position and impressions to raise health score," which is circular advice: those ARE the score. This is the same shortcut my own `w05` decision tree took with `imp_late` — a model finding the label's own construction instead of a genuine pattern.

**Finding 2 — "What Predicts Growth?" (Logistic Regression, 71% holdout accuracy, p.29)**
Two questions the write-up doesn't answer. First: what's the base rate? 71% accuracy on a "growing vs declining" split only means something next to the naive floor — if roughly 70% of sampled pages are naturally growing (plausible, given the report's own headline shows +70.6% impressions 30-day trend), 71% could be barely above predicting the majority class every time — the same check I ran against `base_rate` all through `w05`. Second: the methodology page describes every model in the ML pipeline as an "80/20 split" with no mention of grouping by brand — but this dataset spans 57 brands. A random row-wise split risks the same client-leakage problem that pushed me toward a size-balanced GroupKFold in `w05`: a model could partly be learning "this brand's typical content" rather than a real growth signal, and a random split wouldn't catch that.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.




## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*


`w05` already used a size-balanced GroupKFold split — that IS the honest version. So the "before" here is the naive version: a plain random 80/20 row-wise split, ignoring `client_hash_id` entirely (the same kind of split the FlyRank paper's methodology page describes for its own models). If the naive split scores noticeably higher than the grouped OOF scores from `w05`, that gap IS the client-leakage effect — a concrete, numeric answer to the methodology question raised in Section 1.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import numpy as np

def build_matrix(frame, columns=None):
    num = frame[numeric_features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
    cat = frame[categorical_features].fillna("unknown").astype(str)
    enc = pd.get_dummies(cat, prefix=categorical_features, dtype=float)
    mat = pd.concat([num.reset_index(drop=True), enc.reset_index(drop=True)], axis=1)
    if columns is not None:
        mat = mat.reindex(columns=columns, fill_value=0)
    return mat

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(y_true)[order].mean())

y = model_df["still_declining_may"].to_numpy()

# NAIVE split: random 80/20, ignoring client_hash_id entirely
rng = np.random.default_rng(42)
shuffled_idx = rng.permutation(len(model_df))
split_point = int(len(model_df) * 0.8)
train_idx, test_idx = shuffled_idx[:split_point], shuffled_idx[split_point:]

train_df, test_df = model_df.iloc[train_idx], model_df.iloc[test_idx]
X_train = build_matrix(train_df)
X_test = build_matrix(test_df, columns=X_train.columns)
y_train, y_test = y[train_idx], y[test_idx]

naive_tree = DecisionTreeClassifier(max_depth=4, min_samples_leaf=200, class_weight="balanced", random_state=42)
naive_tree.fit(X_train, y_train)
naive_scores = naive_tree.predict_proba(X_test)[:, 1]

print("Client overlap check — how much train/test overlap does the naive split allow?")
train_clients = set(train_df["client_hash_id"])
test_clients = set(test_df["client_hash_id"])
print(f"Clients in both train AND test: {len(train_clients & test_clients)} / {model_df['client_hash_id'].nunique()}")

print("\nBEFORE (naive 80/20, no grouping) vs AFTER (w05's size-balanced GroupKFold OOF):")
for k in [20, 50, 100, 200]:
    naive_p = precision_at_k(y_test, naive_scores, min(k, len(y_test)))
    honest_p = precision_at_k(y, oof_scores["decision_tree"], k)
    print(f"  k={k:>3}: naive={naive_p:.3f}   honest(grouped OOF)={honest_p:.3f}   gap={naive_p - honest_p:+.3f}")


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*




Restating what `w03_feature_leakage_check.ipynb` already established, applied to the final feature set actually used in `w05`'s model:

- **Excluded, confirmed:** `content_updated_date`, `last_optimized_date` (snapshot-table dates that don't reliably reflect pre-decision-moment state — see `w03` for the full trap demonstration), `optimization_eligible_date` (forward-looking product field), `is_published`/`is_deleted` (product state, not a search signal), any `fact_content_daily_performance` row after `2026-04-30` (May/June — used only as the check-only label source, never a feature).
- **Included, and why each is safe:** `imp_early`, `imp_late`, `impressions_90d`, `clicks_90d`, `avg_position_90d`, `ctr` — all observed strictly within the `2026-01-30`–`2026-04-30` feature window. `content_type`, `main_intent`, `word_count`, `char_count`, `competition_level`, `search_volume` — `dim_content` metadata, stable content properties rather than time-sensitive state.
- **The one caveat this course of notebooks surfaced that's worth restating here:** even without any literal future-window or product-flag leakage, `w05`'s Section 4 found that `imp_late` (and, in the ablation, `impressions_90d`) let the tree take a shortcut through the label's own construction (`still_declining_may` has a floor effect when the late-window count is already near zero) — a leakage-*adjacent* problem, not a column that shouldn't have been there, but a label built in a way that a subset of legitimate features can trivially satisfy. Worth flagging as the more subtle lesson from this whole build: passing the "is this column from the future / from a product flag" checklist doesn't guarantee a feature can't still shortcut a poorly-constructed label.

In [ ]:
# Confirm no banned columns made it into the final feature set
banned = {"content_updated_date", "last_optimized_date", "optimization_eligible_date",
          "is_published", "is_deleted", "impressions_may", "rate_may", "still_declining_may",
          "content_hash_id", "client_hash_id", "is_declining"}
final_features = set(numeric_features) | set(categorical_features)
leaked = final_features & banned
print(f"Final feature set: {sorted(final_features)}")
print(f"\nAny banned columns present? {bool(leaked)}  ({leaked if leaked else 'none'})")

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

*(Pick your boldest sentence from `w04` or `w05` — a strong candidate is w05's Section 3 headline about the decision tree beating the baseline — and rewrite it here. Draft below, adjust to whichever sentence you actually want to soften.)*

**Before:** "The decision tree beats the baseline at every K, proving it's a better prioritization tool."

**After:** "Within this size-balanced client-grouped validation, the decision tree's precision@K was measured to be higher than the recomputed baseline rule at every K tested — but permutation importance showed this was driven almost entirely by a single feature tied mechanically to how the check-only label was constructed, not by a broader content-quality signal. The result is directional evidence that a learned model can outperform the hand-set rule on this exact label, and decision-support for further investigation with a better-constructed label — not proof the model captures real content decline.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.